# Evaluation of the three demos
Run the models on the CIFAR10 test set.

Each model consists of a pretrained backbone (trained using pretext-tasks, self-supervised) and a head (linear probe) trained online (same time as backbone) for image classification of the CIFAR10's 10 classes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/TP5-SSL/cs294-158-ssl
!pip install -q datasets # needed for HuggingFace
!pip install warmup-scheduler
!pip install -e .

/content/drive/MyDrive/TP5-SSL/cs294-158-ssl
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for warmup-scheduler: filename=warmup_scheduler-0.3-py3-none-any.whl size=2983 sha256=e11411f276ae2d80c096df82229642d4b4c1b988d922c9846010799e329ff9f3
  Stored in directory: /root/.cache/pip/wheels/1c/a4/09/f426ccd92c082c8e345da4ed5ef5c9d3b076058739507a95ef
Successfully built warmup-scheduler
Obtaining file:///content/drive/MyDrive/TP5-SSL/cs294-158-ssl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for deepul_helper (pyproject.toml) ... done
  Created wheel for deepul_helper: filename=deepul_helper-0.1.0-0.editable-py3-none-any.whl size=2758 sha256=95480230cdddace8c9cb71b9c2daca5da78f5b32f670519c43eb50ddf2662169
  

In [3]:
data_dir = '/content/drive/MyDrive/TP5-SSL/cs294-158-ssl/data'

## Evaluation on CIFAR10

Answers Question 3.

Run the script to evaluate on test sets (CIFAR10) for the three pretext tasks.

In [ ]:
# Evaluation on CIFAR10
!python scripts/eval_classif.py

{'Method': 'context_encoder', 'Params_M': 71.1, 'Top1': 45.81, 'Top5': 90.28}
{'Method': 'rotation', 'Params_M': 1.4, 'Top1': 79.91, 'Top5': 99.12}
{'Method': 'simclr', 'Params_M': 23.8, 'Top1': 92.83, 'Top5': 99.87}
{'Method': 'spatial_context', 'Params_M': 1.9, 'Top1': 42.88, 'Top5': 88.4}

cifar10 test accuracy (%)
         Method  Params_M  Top1  Top5
context_encoder      71.1 45.81 90.28
       rotation       1.4 79.91 99.12
         simclr      23.8 92.83 99.87
spatial_context       1.9 42.88 88.40


## Evaluation on another Dataset: Tiny ImageNet
This dataset is harder since it has 200 classes.    
We use a stratified subset of the dataset with a total of 50k images for training.    
Due to compute limitation, we choose 50 epochs for each model (accuracy will be lower) and a batch size of 128.

**Download Tiny ImageNet (done only once)**        
Download and create the stratified subset. With 200 classes, we will have 250 image per class.

In [ ]:
# DONE ONLY ONCE
# Download Tiny ImageNet to the local Colab disk (/content), not to Drive:
# reading ~100k small JPEGs through the Drive mount is extremely slow.
import os
import numpy as np
from datasets import load_dataset

hf = load_dataset('zh-plus/tiny-imagenet')
print(hf)  # check the split names ('train' and 'valid' assumed below)

n_classes = hf['train'].features['label'].num_classes
targets = np.array(hf['train']['label'])
rng = np.random.RandomState(0)
idx = np.concatenate([rng.choice(np.where(targets == c)[0], 250, replace=False)
                      for c in range(n_classes)])  # 250 per class = 50k

def to_arrays(split):
    xs, ys = [], []
    for ex in split:
        xs.append(np.array(ex['image'].convert('RGB')))
        ys.append(ex['label'])
    return np.stack(xs), np.array(ys)

xtr, ytr = to_arrays(hf['train'].select(idx))
xte, yte = to_arrays(hf['valid'])

os.makedirs(f'{data_dir}/tiny', exist_ok=True)
np.savez(f'{data_dir}/tiny/tiny_subset.npz', xtr=xtr, ytr=ytr, xte=xte, yte=yte)
print(xtr.shape, xte.shape)  # expect (50000, 64, 64, 3) (10000, 64, 64, 3)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 100000
    })
    valid: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})
(50000, 64, 64, 3) (10000, 64, 64, 3)


## Training on Tiny ImageNet
Train the three models.   
Checkpoints are written to `results/tiny_<task>/` (on Drive)

In [ ]:
# Rotation
!python train_self_supervised_task.py -d tiny -t rotation -b 128 -e 50 -o sgd --lr 0.1 -w 5e-4 -u 0
# for training logs, see `logs/rotation_tiny.txt`

In [ ]:
# SimCLR (-r to resume)
!python train_self_supervised_task.py -d tiny -t simclr -b 128 -e 50 -o lars --lr 1.0 -w 1e-6 -u 0 -r
# for training logs, see `logs/simclr_tiny.txt`

In [ ]:
# Context encoder
!python train_self_supervised_task.py -d tiny -t context_encoder -b 128 -e 50 -o adam --lr 1e-3 -w 0 -u 0 -r
# for training logs, see `logs/context_encoder_tiny.txt`

In [ ]:
# Spatial context on Tiny ImageNet
!python train_self_supervised_task.py -d tiny -t spatial_context -b 128 -e 50 -o sgd --lr 0.1 -w 5e-4 -u 0
# for training logs, see `logs/spatial_context_tiny.txt`

In [ ]:
# Spatial context on tested on Tiny ImageNet with larger batch size
!python train_self_supervised_task.py -d tiny -t spatial_context -b 512 -e 50 -o sgd --lr 0.2 -w 5e-4 -u 0
# for training logs, see `logs/spatial_context_tiny_batch512.txt`

## Evaluation on Tiny ImageNet

In [ ]:
# Evaluate on Tiny ImageNet
!python scripts/eval_classif.py tiny

{'Method': 'context_encoder', 'Params_M': 71.1, 'Top1': 5.59, 'Top5': 17.58}
{'Method': 'rotation', 'Params_M': 1.4, 'Top1': 23.5, 'Top5': 46.86}
{'Method': 'simclr', 'Params_M': 23.8, 'Top1': 15.9, 'Top5': 39.04}
{'Method': 'spatial_context', 'Params_M': 1.9, 'Top1': 3.91, 'Top5': 12.09}

tiny test accuracy (%)
         Method  Params_M  Top1  Top5
context_encoder      71.1  5.59 17.58
       rotation       1.4 23.50 46.86
         simclr      23.8 15.90 39.04
spatial_context       1.9  3.91 12.09


## Evaluation on a 3rd dataset (STL-10)

In [ ]:
%cd /content/drive/MyDrive/TP5-SSL/cs294-158-ssl

/content/drive/MyDrive/TP5-SSL/cs294-158-ssl


**Download the dataset and create the subset**      
Create a subset of 45k unlabeled images, and keep the 5k labeled images.        
Save the results as numpy arrays.  

In [ ]:
# DONE ONLY ONCE
# Download STL-10, create a subset of 45k unlabeled images, and save to disk.
# Save the results as numpy arrays to avoid slow reading of many small JPEGs through the Drive mount.
from datasets import load_dataset
import os
import numpy as np
from PIL import Image

hf = load_dataset('jxie/stl10', cache_dir=f'{data_dir}/hf_cache')
print(hf)  # check which splits exist

def to_arrays(split):
    xs, ys = [], []
    for ex in split:
        xs.append(np.array(ex['image'].convert('RGB')))
        ys.append(ex.get('label', -1))
    return np.stack(xs), np.array(ys, dtype=np.int64)

unlab = hf['unlabeled']
assert len(unlab) == 100000, len(unlab)
idx = np.sort(np.random.RandomState(0).choice(len(unlab), 45000, replace=False))

x_lab, y_lab = to_arrays(hf['train'])
x_unlab, _ = to_arrays(unlab.select(idx))
xte, yte = to_arrays(hf['test'])
assert y_lab.min() == 0 and y_lab.max() == 9 and yte.min() == 0 and yte.max() == 9

xtr = np.concatenate([x_lab, x_unlab])
ytr = np.concatenate([y_lab, np.full(len(x_unlab), -1, dtype=np.int64)])

os.makedirs(f'{data_dir}/stl10', exist_ok=True)
np.save(f'{data_dir}/stl10/train_x.npy', xtr)
np.save(f'{data_dir}/stl10/train_y.npy', ytr)
np.save(f'{data_dir}/stl10/test_x.npy', xte)
np.save(f'{data_dir}/stl10/test_y.npy', yte)
print(xtr.shape, xte.shape)  # expect (50000, 96, 96, 3) (8000, 96, 96, 3)

Image.fromarray(xtr[0]).save('stl10_check.png')

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 8000
    })
    train_0: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_1: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_2: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_3: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_4: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_5: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_6: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_7: Dataset({
        features: ['image', 'label'],
        num_rows: 1000
    })
    train_8: Dataset({
        features: ['image', 'label'],
        num_rows: 10

### Train on STL-10
**Spatial Context**

In [ ]:
# Spatial context on STL-10
!python train_self_supervised_task.py -d stl10 -t spatial_context -b 128 -e 50 -o sgd --lr 0.1 -w 5e-4 -u 0 -r
# for training logs, see `logs/spatial_context_stl10.txt`

In [ ]:
# Rotation on STL-10 for comparison
!python train_self_supervised_task.py -d stl10 -t rotation -b 128 -e 50 -o sgd --lr 0.05 -w 5e-4 -u 0 -r
# for training logs, see `logs/rotation_stl10.txt`

In [ ]:
# Train rotation on CIFAR-10 with less epochs for a fair comparison with spatial context
!python train_self_supervised_task.py -d cifar10 -t rotation -b 128 -e 50 -o sgd --lr 0.1 -w 5e-4 -u 0

### Evaluation on Test set

In [ ]:
!python scripts/eval_classif.py stl10